# 21.07 - Temporal models overview

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Video model strategy note.

Compare recurrent frame-feature models, 3D CNNs, and pooled frame predictions, then implement small shape-safe baselines.

## Core Ideas

A video batch normally has shape `[B, T, C, H, W]`. A 2D CNN can encode each frame into `[B, T, F]`; an LSTM or GRU then models the ordered feature sequence. This is modular and memory-friendly, but recurrent processing is partly sequential.

A 3D CNN receives `[B, C, T, H, W]` and learns spatial-temporal filters jointly. It exposes short motion patterns directly, but activation memory grows quickly with clip length and spatial size. A frame-logit pooling model is fastest and easiest to debug, but discards ordering.

Clip length is a resource decision: longer clips cover more context, while shorter clips allow larger batches or higher spatial resolution. Always document the tensor layout before passing data between video components.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## Exercise 21-A: Compare three temporal strategies

Write a compact strategy table that makes the runtime/accuracy tradeoffs explicit. `activation_values` is a rough count for comparison, not a framework memory profiler.

**Return structure — `estimate_temporal_tradeoffs`:** returns a `list[dict]` of length 3. Each dictionary has exactly four keys: `name` (`str`), `activation_values` (`int`, non-negative), `temporal_mechanism` (`str`), and `tradeoff` (`str`). The three names are `frame_pooling`, `cnn_gru`, and `3d_cnn`, in that order.

In [ ]:
# TODO 21-A
def estimate_temporal_tradeoffs(batch_size, clip_length, feature_dim, height, width):
    # TODO: return the three strategy dictionaries described above.
    raise NotImplementedError('Complete Exercise 21-A')


# Smoke check: run this after implementing the function above.
strategy_rows = estimate_temporal_tradeoffs(2, 8, 32, 16, 16)
print(strategy_rows)

## Exercise 21-B: CNN + GRU pseudobaseline

Encode every frame with the same 2D CNN, preserve the time dimension, feed the feature sequence to a GRU, and classify from its final output.

**Return structure — `CNNGRUClassifier`:** constructing the class returns an `nn.Module` instance. Calling it with a floating-point `torch.Tensor` shaped `[B, T, C, H, W]` on the same device as the module returns a floating-point `torch.Tensor` of logits shaped `[B, K]` on that device, where `K=num_classes`.

In [ ]:
# TODO 21-B
class CNNGRUClassifier(nn.Module):
    def __init__(self, in_channels=3, feature_dim=32, hidden_dim=24, num_classes=4):
        super().__init__()
        # TODO: define a shared frame encoder, a batch-first GRU, and a classifier.
        raise NotImplementedError('Complete CNNGRUClassifier.__init__')

    def forward(self, videos):
        # TODO: [B,T,C,H,W] -> [B*T,C,H,W] -> [B,T,F] -> [B,K].
        raise NotImplementedError('Complete CNNGRUClassifier.forward')


# Smoke check: run this after implementing the class above.
gru_model = CNNGRUClassifier().to(DEVICE)
smoke_videos = torch.randn(2, 5, 3, 16, 16, device=DEVICE)
smoke_gru_logits = gru_model(smoke_videos)
print('CNN+GRU logits:', smoke_gru_logits.shape, smoke_gru_logits.dtype, smoke_gru_logits.device)

## Exercise 21-C: Tiny 3D CNN

Build a small 3D convolutional alternative. Remember that `Conv3d` expects channels before time, so the input layout must change from `[B,T,C,H,W]` to `[B,C,T,H,W]`.

**Return structure — `Tiny3DCNN`:** constructing the class returns an `nn.Module` instance. Calling it with a floating-point `torch.Tensor` shaped `[B, T, C, H, W]` on the module device returns floating-point logits shaped `[B, K]` on that device, where `K=num_classes`.

In [ ]:
# TODO 21-C
class Tiny3DCNN(nn.Module):
    def __init__(self, in_channels=3, hidden_channels=12, num_classes=4):
        super().__init__()
        # TODO: define Conv3d feature extraction, global pooling, and classification.
        raise NotImplementedError('Complete Tiny3DCNN.__init__')

    def forward(self, videos):
        # TODO: permute to [B,C,T,H,W] before the 3D convolution.
        raise NotImplementedError('Complete Tiny3DCNN.forward')


# Smoke check: run this after implementing the class above.
conv3d_model = Tiny3DCNN().to(DEVICE)
smoke_3d_logits = conv3d_model(smoke_videos)
print('3D CNN logits:', smoke_3d_logits.shape, smoke_3d_logits.dtype, smoke_3d_logits.device)

## Test Cases

Run this cell after completing the TODO cells. A correct implementation prints `Day 21 tests passed`.

**Return structure — `run_day21_tests`:** returns `None`. Success is communicated by completing all assertions and printing exactly `Day 21 tests passed`; a failed requirement raises `AssertionError`.

In [ ]:
def run_day21_tests():
    assert 'estimate_temporal_tradeoffs' in globals()
    rows = estimate_temporal_tradeoffs(2, 8, 32, 16, 16)
    assert isinstance(rows, list) and len(rows) == 3
    assert [row['name'] for row in rows] == ['frame_pooling', 'cnn_gru', '3d_cnn']
    assert all(set(row) == {'name', 'activation_values', 'temporal_mechanism', 'tradeoff'} for row in rows)
    assert all(isinstance(row['activation_values'], int) and row['activation_values'] >= 0 for row in rows)

    videos = torch.randn(2, 5, 3, 16, 16, device=DEVICE)
    gru = CNNGRUClassifier(num_classes=4).to(DEVICE)
    gru_logits = gru(videos)
    assert gru_logits.shape == (2, 4) and gru_logits.dtype == torch.float32
    assert gru_logits.device == DEVICE

    conv3d = Tiny3DCNN(num_classes=4).to(DEVICE)
    conv3d_logits = conv3d(videos)
    assert conv3d_logits.shape == (2, 4) and conv3d_logits.dtype == torch.float32
    assert conv3d_logits.device == DEVICE
    assert sum(p.numel() for p in gru.parameters()) > 0
    assert sum(p.numel() for p in conv3d.parameters()) > 0
    print('Day 21 tests passed')


run_day21_tests()

## Day 21 Checklist

- [ ] I can state the expected layout for 2D-frame, recurrent, and 3D-convolution components.
- [ ] I can explain when frame pooling, CNN+GRU, or a 3D CNN is a sensible baseline.
- [ ] I checked output shape, dtype, and device for both implemented models.
- [ ] I can explain why clip length affects context, activation memory, and throughput.
- [ ] `run_day21_tests()` prints the required pass message.